# <span style="color:orange">Notebook 4 (bonus): The live digital twin</span>

<span style="color:orange">Welcome to notebook 4. Notebooks 1 to 3 replay the simulations of the twin; here, OpenDT runs live, as in the demo on the projector: its services start in Docker, the twin ingests the telemetry window by window, and Grafana shows what it predicts. In this notebook, you will learn:</span>
- <span style="color:orange">how to start OpenDT's live twin with Docker, and what each of its services does;</span>
- <span style="color:orange">how to read the twin's predictions from its REST API;</span>
- <span style="color:orange">how to change the twin while it runs (e.g., its hosts), and follow the Self-Calibrator in its logs.</span>

<span style="color:orange">By the end of the notebook, you will be able to run OpenDT on your own machine, and steer a running digital twin, as its human-in-the-loop.</span>

# Bonus: The Live Digital Twin

*Optional, after class; requires Docker (about 15 minutes).*

We implement OpenDT as a Docker Compose microservice system in which the core components (data source, simulator,
calibrator, and API) communicate through Apache Kafka.
As in the ICPE demo, we start it with `make up`, change the hosts from the REST API, follow the dashboard in Grafana,
and see how calibration works, with investigation in the logs.

![High-level overview of OpenDT](resources/images/opendt_design.png)

*High-level overview of OpenDT (OpenDT paper, Figure 2).*

The `make` commands go in a terminal, in the OpenDT folder (the one with the `Makefile`), after the setup in the
README.

## 1. Starting the Live Twin

```
make up config=tutorial/resources/configs/live_twin.yaml
```

`make up` first deletes the Kafka and Grafana volumes of the previous run; the first start also builds the Docker
images, for a few minutes.
OpenDT supports a configurable acceleration factor, expressed as a ratio between simulation and real-world time; this
configuration uses the maximum acceleration allowed by computational resources (`speed_factor: -1`), with windows of
operation of 5 minutes.
Kafka needs port 9092 on the computer, so a local Kafka (or any other program on that port) has to stop first.

The dashboard is at http://localhost:3000, on its Detailed screen: Week, Day, or 6 h (top left) choose the time it
shows, and its Tutorial screen goes through the three use cases on the running twin.

![The OpenDT dashboard in Grafana](resources/images/grafana_dashboard.png)

*<s>The OpenDT dashboard: actual vs. simulated power draw (top) and carbon emissions (bottom).</s> <span style="color:orange">The OpenDT dashboard, on its Detailed screen: the view and the twin's levers on top, its numbers now, then the plots by category.</span>*

## 2. Predictions from the REST API

The same predictions come from the REST API (documentation at http://localhost:3001/docs), here in 5-minute samples;
until the first window of operation is done, the API answers 404.

In [ ]:
import json
import urllib.request

import pandas as pd
from resources import PATHS, ask_question, legend_above

API = "http://localhost:3001"

with urllib.request.urlopen(f"{API}/api/power?interval_seconds=300") as response:
    power = pd.DataFrame(json.load(response)["data"])
power = power.set_index(pd.to_datetime(power["timestamp"]))
power = power.rename(columns={"actual_power": "Ground Truth", "simulated_power": "OpenDT"})

ax = (power[["Ground Truth", "OpenDT"]] / 1000).plot(color=["#999999", "#009E73"], style=["-", "-"])
ax.set(xlabel="Time [UTC]", ylabel="Power Draw [kW]", ylim=(0, None))
legend_above(ax)

## 3. Changing the Hosts from the REST API

`resources/configs/topology_2770_hosts.json` is the SURF topology of OpenDT with 10x the hosts (2,770).

In [ ]:
ask_question("4.1")

In [ ]:
body = (PATHS.root / "tutorial/resources/configs/topology_2770_hosts.json").read_bytes()
request = urllib.request.Request(
    f"{API}/api/topology", data=body, method="PUT", headers={"Content-Type": "application/json"}
)
with urllib.request.urlopen(request) as response:
    print(json.load(response))

## 4. Calibration, with Investigation in the Logs

```
make down
make up config=tutorial/resources/configs/live_twin_calibration.yaml
make logs-calibrator
```

After each grid search, the log shows the MAPE of each candidate r, then `Best: cpuPowerModel.calibrationFactor=...` and
`Published topology with ...`; Ctrl+C stops following the log.

In [ ]:
ask_question("4.2")

## 5. Stopping the Live Twin

```
make down
```

All services mount a shared host directory that serves as a file-based workspace for configuration files and
simulation outputs; for each run, it is `data/<run id>/`.

*Back to:* [README.md](README.md).